In [11]:
import cv2
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

In [12]:
# 1. Konfigurera med RunningMode.VIDEO
base_options = python.BaseOptions(model_asset_path='face_landmarker.task')
options = vision.FaceLandmarkerOptions(
    base_options=base_options,
    output_face_blendshapes=True,
    running_mode=vision.RunningMode.VIDEO
)
detector = vision.FaceLandmarker.create_from_options(options)

video_path = "data/test_video.mov"
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
total_frames = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

max_frames = 100
frame_indices = np.linspace(0, total_frames - 1, max_frames, dtype=int)

In [13]:

features = []
valid_indices = []

for idx in frame_indices:
    cap.set(cv2.CAP_PROP_POS_FRAMES, idx)
    ret, frame = cap.read()
    if not ret:
        continue
        
    # Beräkna tidsstämpel i millisekunder baserat på bildrutan och FPS
    timestamp_ms = int((idx / fps) * 1000)
    
    rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=rgb_frame)
    
    # Skicka med tidsstämpeln när läget är VIDEO
    result = detector.detect_for_video(mp_image, timestamp_ms)
    
    if result.face_blendshapes:
        blendshapes = [score.score for score in result.face_blendshapes[0]]
        features.append(blendshapes)
        valid_indices.append(idx)

cap.release()
X = np.array(features)
print(f"Hittade ansikte i {len(X)} av {max_frames} samplade bildrutor.")

Hittade ansikte i 4 av 100 samplade bildrutor.


In [15]:
pca = PCA(n_components=10)
X_pca = pca.fit_transform(X)

n_clusters = 5
kmeans = KMeans(n_clusters=n_clusters, random_state=42, n_init=10)
labels = kmeans.fit_predict(X_pca)

ValueError: n_components=10 must be between 0 and min(n_samples, n_features)=4 with svd_solver='full'

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(X_pca[:, 0], X_pca[:, 1], c=labels, cmap='tab10', alpha=0.6)
plt.title("Klustring av ansiktsuttryck")
plt.xlabel("PCA 1")
plt.ylabel("PCA 2")
plt.colorbar(label='Kluster')
plt.show()

NameError: name 'X_pca' is not defined

<Figure size 1000x600 with 0 Axes>